In [62]:
import functions
import importlib
importlib.reload(functions)

from functions import *

In [50]:
len(frames(1,3))

37500

In [63]:
import numpy as np

C_MPS = 299_792_458.0
L1_HZ = 1575.42e6
CA_CHIP_HZ = 1.023e6

def sample_chip_sequence_to_iq_file(
    chips_01,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=CA_CHIP_HZ,
    fs=4.0e6,
    f_if=1.25e6,
    doppler_hz=0.0,
    distance_m=0.0,
    phase0=0.0,
    chunk_size=10_000_000,
):
    chips_01 = np.asarray(chips_01, dtype=np.uint8)
    if not np.all((chips_01 == 0) | (chips_01 == 1)):
        raise ValueError("chips_01 must contain only 0/1")

    chips = np.where(chips_01 == 0, -1.0, 1.0).astype(np.float32)

    signal_duration = len(chips) / chip_rate
    n_signal_samples = int(np.floor(signal_duration * fs))

    delay_s = distance_m / C_MPS
    delay_samples = int(np.round(delay_s * fs))
    n_total_samples = n_signal_samples + delay_samples

    # Match code rate to carrier Doppler approximately
    chip_rate_eff = chip_rate * (1.0 + doppler_hz / L1_HZ)

    with open(out_file, "wb") as f:
        for start in range(0, n_total_samples, chunk_size):
            stop = min(start + chunk_size, n_total_samples)
            sample_idx = np.arange(start, stop, dtype=np.int64)

            src_sample_idx = sample_idx - delay_samples
            valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)

            i = np.zeros(stop - start, dtype=np.float32)
            q = np.zeros(stop - start, dtype=np.float32)

            if np.any(valid):
                src_t = src_sample_idx[valid].astype(np.float64) / fs

                chip_index = np.floor(src_t * chip_rate_eff).astype(np.int64)
                chip_index = np.clip(chip_index, 0, len(chips) - 1)
                baseband = chips[chip_index]

                t = sample_idx[valid].astype(np.float64) / fs
                phase = 2.0 * np.pi * (f_if + doppler_hz) * t + phase0

                i[valid] = (baseband * np.cos(phase)).astype(np.float32)
                q[valid] = (baseband * np.sin(phase)).astype(np.float32)

            iq = np.empty(2 * len(i), dtype=np.float32)
            iq[0::2] = i
            iq[1::2] = q
            iq.tofile(f)

    return n_total_samples


sliced = round(len(modulo2_frames_runs(1, Z_count_start, 1)) / 8)
spread_bits = res[:sliced]

chips_01 = np.frombuffer(spread_bits.unpack(), dtype=np.uint8)

n_samples = sample_chip_sequence_to_iq_file(
    chips_01,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    f_if=0,#1.25e6,
    doppler_hz=200.0,        # example Doppler
    distance_m=0,  # example range
    phase0=0.3,
)

print("chips:", len(chips_01))
print("complex samples:", n_samples)

KeyboardInterrupt: 

In [64]:
import numpy as np

C_MPS = 299_792_458.0  # speed of light [m/s]


def sample_chip_sequences_to_iq_file(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz=None,
    phase0=None,
    amplitudes=None,
    use_relative_delays=True,
    common_delay_s=0.0,
    chunk_size=10_000_000,
):
    """
    Multi-satellite complex baseband IQ generator.

    Writes interleaved float32 IQ samples:
        I0, Q0, I1, Q1, I2, Q2, ...

    Parameters
    ----------
    chip_streams_01 : list of arrays
        One 0/1 chip stream per satellite.
    distances_m : array-like
        One distance per satellite [m].
    out_file : str
        Output filename.
    chip_rate : float
        Chip rate [Hz].
    fs : float
        Sample rate [Hz].
    doppler_hz : array-like or None
        Per-satellite Doppler [Hz]. Defaults to zeros.
    phase0 : array-like or None
        Per-satellite initial carrier phase [rad]. Defaults to zeros.
    amplitudes : array-like or None
        Per-satellite amplitudes. Defaults to ones.
    use_relative_delays : bool
        If True, subtract the minimum geometric delay so the earliest
        satellite starts at t=0, then add common_delay_s.
    common_delay_s : float
        Extra delay added to all satellites. Useful if you want all
        pseudoranges shifted upward by a common amount.
    chunk_size : int
        Samples per write chunk.

    Returns
    -------
    n_total_samples : int
        Total number of complex samples written.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    n_sats = len(chip_streams_01)

    if len(distances_m) != n_sats:
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams from 0/1 to -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    signal_duration_s = n_chips / chip_rate
    n_signal_samples = int(np.floor(signal_duration_s * fs))

    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    delays_s = delays_s + float(common_delay_s)
    delay_samples = np.round(delays_s * fs).astype(np.int64)

    if doppler_hz is None:
        doppler_hz = np.zeros(n_sats, dtype=np.float64)
    else:
        doppler_hz = np.asarray(doppler_hz, dtype=np.float64)
        if len(doppler_hz) != n_sats:
            raise ValueError("doppler_hz must have same length as chip_streams_01")

    if phase0 is None:
        phase0 = np.zeros(n_sats, dtype=np.float64)
    else:
        phase0 = np.asarray(phase0, dtype=np.float64)
        if len(phase0) != n_sats:
            raise ValueError("phase0 must have same length as chip_streams_01")

    if amplitudes is None:
        amplitudes = np.ones(n_sats, dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != n_sats:
            raise ValueError("amplitudes must have same length as chip_streams_01")

    max_delay_samples = int(np.max(delay_samples))
    n_total_samples = n_signal_samples + max_delay_samples

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Delay samples:", delay_samples)
    print("Doppler [Hz]:", doppler_hz)
    print("Signal samples (undelayed):", n_signal_samples)
    print("Total complex samples:", n_total_samples)

    with open(out_file, "wb") as f:
        for start in range(0, n_total_samples, chunk_size):
            stop = min(start + chunk_size, n_total_samples)
            sample_idx = np.arange(start, stop, dtype=np.int64)

            i_sum = np.zeros(stop - start, dtype=np.float32)
            q_sum = np.zeros(stop - start, dtype=np.float32)

            for s in range(n_sats):
                src_sample_idx = sample_idx - delay_samples[s]
                valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)

                if not np.any(valid):
                    continue

                # Sample the chip stream
                chip_index = np.floor(
                    src_sample_idx[valid].astype(np.float64) * chip_rate / fs
                ).astype(np.int64)
                chip_index = np.clip(chip_index, 0, n_chips - 1)
                baseband = chips[s][chip_index].astype(np.float32)

                # Complex baseband carrier with Doppler only
                t = sample_idx[valid].astype(np.float64) / fs
                phase = 2.0 * np.pi * doppler_hz[s] * t + phase0[s]

                i_sum[valid] += amplitudes[s] * baseband * np.cos(phase).astype(np.float32)
                q_sum[valid] += amplitudes[s] * baseband * np.sin(phase).astype(np.float32)

            iq = np.empty(2 * (stop - start), dtype=np.float32)
            iq[0::2] = i_sum
            iq[1::2] = q_sum
            iq.tofile(f)

    return n_total_samples

In [74]:
import numpy as np

C_MPS = 299_792_458.0  # speed of light [m/s]


def sample_chip_sequences_to_iq_file_variable_doppler(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz_chunks=None,
    phase0=None,
    amplitudes=None,
    use_relative_delays=True,
    common_delay_s=0.0,
    chunk_size=10_000_000,
):
    """
    Multi-satellite complex baseband IQ generator with chunk-varying Doppler.

    Writes interleaved float32 IQ samples:
        I0, Q0, I1, Q1, I2, Q2, ...

    Parameters
    ----------
    chip_streams_01 : list of arrays
        One 0/1 chip stream per satellite.
    distances_m : array-like
        One distance per satellite [m].
    out_file : str
        Output filename.
    chip_rate : float
        Chip rate [Hz].
    fs : float
        Sample rate [Hz].
    doppler_hz_chunks : array-like or None
        Doppler schedule per chunk, shape:
            (n_sats, n_chunks)
        or
            (n_chunks, n_sats)
        If None, all Dopplers are zero.
    phase0 : array-like or None
        Per-satellite initial carrier phase [rad]. Defaults to zeros.
    amplitudes : array-like or None
        Per-satellite amplitudes. Defaults to ones.
    use_relative_delays : bool
        If True, subtract the minimum geometric delay so the earliest
        satellite starts at t=0, then add common_delay_s.
    common_delay_s : float
        Extra delay added to all satellites.
    chunk_size : int
        Samples per write chunk.

    Returns
    -------
    n_total_samples : int
        Total number of complex samples written.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    n_sats = len(chip_streams_01)

    if len(distances_m) != n_sats:
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams from 0/1 to -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    signal_duration_s = n_chips / chip_rate
    n_signal_samples = int(np.floor(signal_duration_s * fs))

    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    delays_s = delays_s + float(common_delay_s)
    delay_samples = np.round(delays_s * fs).astype(np.int64)

    if phase0 is None:
        phase0 = np.zeros(n_sats, dtype=np.float64)
    else:
        phase0 = np.asarray(phase0, dtype=np.float64)
        if len(phase0) != n_sats:
            raise ValueError("phase0 must have same length as chip_streams_01")

    if amplitudes is None:
        amplitudes = np.ones(n_sats, dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != n_sats:
            raise ValueError("amplitudes must have same length as chip_streams_01")

    max_delay_samples = int(np.max(delay_samples))
    n_total_samples = n_signal_samples + max_delay_samples

    # Number of output chunks
    n_chunks = (n_total_samples + chunk_size - 1) // chunk_size

    # Doppler schedule
    if doppler_hz_chunks is None:
        doppler_hz_chunks = np.zeros((n_sats, n_chunks), dtype=np.float64)
    else:
        doppler_hz_chunks = np.asarray(doppler_hz_chunks, dtype=np.float64)

        if doppler_hz_chunks.ndim != 2:
            raise ValueError("doppler_hz_chunks must be a 2D array")

        # Accept either (n_sats, n_chunks) or (n_chunks, n_sats)
        if doppler_hz_chunks.shape == (n_sats, n_chunks):
            pass
        elif doppler_hz_chunks.shape == (n_chunks, n_sats):
            doppler_hz_chunks = doppler_hz_chunks.T
        else:
            raise ValueError(
                f"doppler_hz_chunks must have shape ({n_sats}, {n_chunks}) "
                f"or ({n_chunks}, {n_sats}), got {doppler_hz_chunks.shape}"
            )

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Delay samples:", delay_samples)
    print("Signal samples (undelayed):", n_signal_samples)
    print("Total complex samples:", n_total_samples)
    print("Chunks:", n_chunks)
    print("Doppler schedule shape:", doppler_hz_chunks.shape)

    # Running carrier phase per satellite, so phase stays continuous across chunks
    phase_acc = phase0.astype(np.float64).copy()

    with open(out_file, "wb") as f:
        for chunk_idx, start in enumerate(range(0, n_total_samples, chunk_size)):
            stop = min(start + chunk_size, n_total_samples)
            n_chunk_samples = stop - start

            sample_idx = np.arange(start, stop, dtype=np.int64)

            i_sum = np.zeros(n_chunk_samples, dtype=np.float32)
            q_sum = np.zeros(n_chunk_samples, dtype=np.float32)

            for s in range(n_sats):
                # This chunk's Doppler for satellite s
                f_dopp = doppler_hz_chunks[s, chunk_idx]

                # Delayed source indices
                src_sample_idx = sample_idx - delay_samples[s]
                valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)

                # Build chunk carrier with continuous phase over the whole chunk
                # phase[n] = phase_acc + 2*pi*f_dopp*n/fs
                if n_chunk_samples > 0:
                    n = np.arange(n_chunk_samples, dtype=np.float64)
                    phase_chunk = phase_acc[s] + 2.0 * np.pi * f_dopp * n / fs
                    cos_chunk = np.cos(phase_chunk).astype(np.float32)
                    sin_chunk = np.sin(phase_chunk).astype(np.float32)

                    # Advance stored phase to next chunk boundary
                    phase_acc[s] = (
                        phase_acc[s]
                        + 2.0 * np.pi * f_dopp * n_chunk_samples / fs
                    ) % (2.0 * np.pi)

                # Before the delayed signal begins, contribution stays zero
                if not np.any(valid):
                    continue

                # Sample the chip stream only where valid
                chip_index = np.floor(
                    src_sample_idx[valid].astype(np.float64) * chip_rate / fs
                ).astype(np.int64)
                chip_index = np.clip(chip_index, 0, n_chips - 1)
                baseband = chips[s][chip_index]

                i_sum[valid] += amplitudes[s] * baseband * cos_chunk[valid]
                q_sum[valid] += amplitudes[s] * baseband * sin_chunk[valid]

            iq = np.empty(2 * n_chunk_samples, dtype=np.float32)
            iq[0::2] = i_sum
            iq[1::2] = q_sum
            iq.tofile(f)

    return n_total_samples

In [78]:
res3=modulo2_frames_runs(3,Z_count_start,3)[:195906250]
res6=modulo2_frames_runs(6,Z_count_start,6)[:195906250]
res9=modulo2_frames_runs(9,Z_count_start,9)[:195906250]
res16=modulo2_frames_runs(16,Z_count_start,16)[:195906250]
res21=modulo2_frames_runs(21,Z_count_start,21)[:195906250]

rho3=np.sqrt(sum((ehpm_to_ECEFlocation(3)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho6=np.sqrt(sum((ehpm_to_ECEFlocation(6)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho9=np.sqrt(sum((ehpm_to_ECEFlocation(9)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho16=np.sqrt(sum((ehpm_to_ECEFlocation(16)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho21=np.sqrt(sum((ehpm_to_ECEFlocation(21)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))

chips_03 = np.frombuffer(res3.unpack(), dtype=np.uint8)
chips_06 = np.frombuffer(res6.unpack(), dtype=np.uint8)
chips_09 = np.frombuffer(res9.unpack(), dtype=np.uint8)
chips_16 = np.frombuffer(res16.unpack(), dtype=np.uint8)
chips_21 = np.frombuffer(res21.unpack(), dtype=np.uint8)

num_sats = 5

chip_streams_01 = [
    chips_03,
    chips_06,
    chips_09,
    chips_16,
    chips_21,
]

distances_m = np.array([
    rho3,
    rho6,
    rho9,
    rho16,
    rho21,
], dtype=float)



df = pd.read_csv("GPS_D1C_doppler.csv")

sat_data = {
    sv: {
        "time": grp["time"].tolist(),
        "doppler": grp["doppler"].tolist()
    }
    for sv, grp in df.groupby("sv")
}

doppler_list = np.array([
    sat_data["G16"]["doppler"][:375],
    sat_data["G16"]["doppler"][:375],
    sat_data["G16"]["doppler"][:375],
    sat_data["G16"]["doppler"][:375],
    sat_data["G16"]["doppler"][:375],
], dtype=float)

phase0 = 2 * np.pi * np.random.rand(num_sats)

amplitudes = np.ones(num_sats, dtype=np.float32) * 0.2

n_total = sample_chip_sequences_to_iq_file_variable_doppler(
    chip_streams_01=chip_streams_01,
    distances_m=distances_m,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz_chunks=doppler_list,   # shape (n_sats, 375) or (375, n_sats)
    phase0=None,
    amplitudes=None,
    use_relative_delays=True,
    common_delay_s=0.0,
    chunk_size=int(37500*20*1023/375),
)

print("complex samples:", n_samples)

Distances [m]: [20054507.79143932 21290782.26494222 23269387.89892859 25244057.78442071
 25542586.34599991]
Delays [s]: [0.         0.00412377 0.01072369 0.01731048 0.01830626]
Delay samples: [    0 16495 42895 69242 73225]
Signal samples (undelayed): 766006842
Total complex samples: 766080067
Chunks: 375
Doppler schedule shape: (5, 375)
complex samples: 766347646


In [ ]:
import numpy as np

C_MPS = 299_792_458.0  # speed of light [m/s]


def sample_chip_sequences_to_if_file(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_if_float32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    f_if=1.25e6,
    amplitudes=None,
    use_relative_delays=True,
    chunk_size=10_000_000,
):
    """
    chip_streams_01: list of chip arrays, each containing 0/1 values
    distances_m: list/array of satellite-to-receiver distances in meters

    Writes real float32 IF samples.

    IMPORTANT:
    - This version applies delay by zero-padding in sample time.
    - No modulo wrap is used for propagation delay.
    - The whole signal is delayed consistently in receive time.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    if len(distances_m) != len(chip_streams_01):
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams 0/1 -> -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    duration = n_chips / chip_rate
    n_signal_samples = int(np.floor(duration * fs))

    if amplitudes is None:
        amplitudes = np.ones(len(chips), dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != len(chips):
            raise ValueError("amplitudes must have same length as chip_streams_01")

    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    # Integer-sample delays for now
    delay_samples = np.round(delays_s * fs).astype(np.int64)
    max_delay = int(np.max(delay_samples))

    # Total output must be long enough to include the latest delayed signal
    n_samples_total = n_signal_samples + max_delay

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Delay samples:", delay_samples)
    print("Total output samples:", n_samples_total)

    with open(out_file, "wb") as f:
        for start in range(0, n_samples_total, chunk_size):
            stop = min(start + chunk_size, n_samples_total)

            sample_idx = np.arange(start, stop, dtype=np.int64)
            t = sample_idx.astype(np.float64) / fs

            carrier = np.cos(2 * np.pi * f_if * t).astype(np.float32)
            signal = np.zeros(stop - start, dtype=np.float32)

            for i in range(len(chips)):
                # Map output sample time back to source sample time
                src_sample_idx = sample_idx - delay_samples[i]

                # Valid only where source time is inside the original waveform
                valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)
                if not np.any(valid):
                    continue

                # Convert valid source sample indices to chip indices
                src_t = src_sample_idx[valid].astype(np.float64) / fs
                chip_phase = src_t * chip_rate
                chip_index = np.floor(chip_phase).astype(np.int64)

                # Clamp to valid chip range
                chip_index = np.clip(chip_index, 0, n_chips - 1)

                baseband = np.zeros(stop - start, dtype=np.float32)
                baseband[valid] = chips[i][chip_index]

                signal += amplitudes[i] * baseband * carrier

            signal.tofile(f)

    return n_samples_total


TypeError: sample_chip_sequence_to_iq_file() got an unexpected keyword argument 'doppler_hz'

In [59]:
res1=modulo2_frames_runs(1,Z_count_start,1)[:95906250]
res2=modulo2_frames_runs(2,Z_count_start,2)[:95906250]
res3=modulo2_frames_runs(3,Z_count_start,3)[:95906250]
res4=modulo2_frames_runs(4,Z_count_start,4)[:95906250]

res6=modulo2_frames_runs(6,Z_count_start,6)[:95906250]
res7=modulo2_frames_runs(7,Z_count_start,7)[:95906250]
res9=modulo2_frames_runs(9,Z_count_start,9)[:95906250]
res11=modulo2_frames_runs(11,Z_count_start,11)[:95906250]

rho1=np.sqrt(sum((ehpm_to_ECEFlocation(1)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho2=np.sqrt(sum((ehpm_to_ECEFlocation(2)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho3=np.sqrt(sum((ehpm_to_ECEFlocation(3)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho4=np.sqrt(sum((ehpm_to_ECEFlocation(4)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho6=np.sqrt(sum((ehpm_to_ECEFlocation(6)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho7=np.sqrt(sum((ehpm_to_ECEFlocation(7)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho9=np.sqrt(sum((ehpm_to_ECEFlocation(9)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho11=np.sqrt(sum((ehpm_to_ECEFlocation(11)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))


chips_01 = np.frombuffer(res1.unpack(), dtype=np.uint8)
chips_02 = np.frombuffer(res2.unpack(), dtype=np.uint8)
chips_03 = np.frombuffer(res3.unpack(), dtype=np.uint8)
chips_04 = np.frombuffer(res4.unpack(), dtype=np.uint8)

chips_06 = np.frombuffer(res6.unpack(), dtype=np.uint8)
chips_07 = np.frombuffer(res7.unpack(), dtype=np.uint8)
chips_09 = np.frombuffer(res9.unpack(), dtype=np.uint8)
chips_11 = np.frombuffer(res11.unpack(), dtype=np.uint8)

distances_m = [
    rho1,  # satellite 1 range to receiver
    rho2,  # satellite 2 range to receiver
    #rho3,  # satellite 3 range to receiver
    #rho4,  # satellite 4 range to receiver
    #rho6,  # satellite 6 range to receiver
    #rho7,  # satellite 7 range to receiver
    #rho9,  # satellite 9 range to receiver
    #rho11,  # satellite 11 range to receiver
]

phase0 = 2 * np.pi * np.random.rand(2)

n_samples = sample_chip_sequences_to_if_file(
    [chips_01, chips_02],# chips_03, chips_04],#chips_06,chips_07,chips_09,chips_11],
    distances_m=distances_m,
    out_file="gps_signal4_if_float32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    f_if=1.25e6,
    doppler_hz=[1500,-2200],
    phase0=phase0,
    amplitudes=[0.4, 0.4], #0.2, 0.2],#0.7, 0.8, 0.6, 0.75],
    use_relative_delays=False,

)

print("samples:", n_samples)

TypeError: sample_chip_sequences_to_if_file() got an unexpected keyword argument 'doppler_hz'

In [60]:
import numpy as np
from scipy.signal import fftconvolve  # [CHANGED] new import — needed for fractional delay convolution

C_MPS = 299_792_458.0  # speed of light [m/s]


# ─────────────────────────────────────────────────────────────────────────────
# [CHANGED] NEW FUNCTION — was not in the original code.
#
# Designs a windowed-sinc FIR filter that shifts a signal by `frac` samples,
# where frac is the sub-sample remainder of the propagation delay.
#
# Why this exists:
#   The original code did np.round(delays_s * fs) which threw away the
#   fractional part of the delay.  For example a true delay of 269520.4
#   samples was stored as 269520, losing 0.4 samples = ~100 ns.  A GNSS
#   receiver's correlator tries to measure delay to ~10 ns, so this error
#   is significant.
#
#   This filter applies the missing fractional shift by convolving the
#   baseband signal with a shifted sinc kernel.  A perfect sub-sample delay
#   in continuous time is multiplication by exp(-j2πf·frac/fs) in the
#   frequency domain, which corresponds to convolution with sinc(n - frac)
#   in the time domain.  A Hann window suppresses Gibbs ringing at the
#   band edges.
#
# Parameters:
#   frac      : fractional delay in [0, 1) samples
#   half_len  : one-sided filter length (total taps = 2*half_len + 1).
#               16 is accurate enough for GPS L1 C/A at 4 MHz; raise to
#               32 for higher sample rates or more demanding use cases.
# ─────────────────────────────────────────────────────────────────────────────
def make_fractional_delay_filter(frac: float, half_len: int = 16) -> np.ndarray:
    if not (0.0 <= frac < 1.0):
        raise ValueError(f"frac must be in [0, 1), got {frac}")

    n = np.arange(-half_len, half_len + 1, dtype=np.float64)
    h = np.sinc(n - frac)               # numpy sinc is normalised: sinc(x) = sin(πx)/(πx)
    window = np.hanning(2 * half_len + 1)
    h *= window
    h /= h.sum()                         # normalise to unity DC gain
    return h.astype(np.float32)


def sample_chip_sequences_to_if_file(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_if_float32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    f_if=1.25e6,
    amplitudes=None,
    use_relative_delays=True,
    chunk_size=10_000_000,
    frac_delay_half_len: int = 16,       # [CHANGED] new parameter — controls filter length.
                                         # Default 16 gives a 33-tap filter.
                                         # Was not present in the original signature.
):
    """
    chip_streams_01: list of chip arrays, each containing 0/1 values
    distances_m: list/array of satellite-to-receiver distances in meters

    Writes real float32 IF samples.

    Changes from original:
    - Propagation delay is now split into integer + fractional parts.
    - The fractional part is applied via a windowed-sinc FIR filter
      (make_fractional_delay_filter) rather than being rounded away.
    - A new parameter `frac_delay_half_len` controls the filter length.
    - Output file is slightly longer to accommodate the filter tail.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    if len(distances_m) != len(chip_streams_01):
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams 0/1 -> -1/+1
    # [UNCHANGED]
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    # [UNCHANGED]
    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    duration = n_chips / chip_rate
    n_signal_samples = int(np.floor(duration * fs))

    # [UNCHANGED]
    if amplitudes is None:
        amplitudes = np.ones(len(chips), dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != len(chips):
            raise ValueError("amplitudes must have same length as chip_streams_01")

    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    # ── [CHANGED] delay splitting ─────────────────────────────────────────────
    #
    # Original code (single line):
    #     delay_samples = np.round(delays_s * fs).astype(np.int64)
    #
    # np.round() discards the sub-sample remainder.  For example:
    #     true delay = 269520.4 samples
    #     after round = 269520          ← 0.4 samples = ~100 ns lost
    #
    # We now split into two parts:
    #   int_delays  — integer sample shift, applied by index arithmetic as before
    #   frac_delays — fractional remainder in [0, 1), applied by the FIR filter
    #
    # np.floor is used instead of np.round so the fractional part is always
    # positive (in [0, 1)).  np.round would give a remainder in [-0.5, 0.5)
    # which complicates the filter design.
    # ─────────────────────────────────────────────────────────────────────────
    exact_delays = delays_s * fs                            # [CHANGED] was: delay_samples = np.round(...)
    int_delays   = np.floor(exact_delays).astype(np.int64) # [CHANGED] integer part of delay
    frac_delays  = exact_delays - int_delays                # [CHANGED] fractional remainder, in [0, 1)

    # [CHANGED] Build one fractional delay filter per satellite, ahead of the
    # chunk loop.  If frac ≈ 0 the filter is nearly a delta function and has
    # negligible effect — no special case needed.
    frac_filters = [
        make_fractional_delay_filter(float(f), half_len=frac_delay_half_len)
        for f in frac_delays
    ]

    # [CHANGED] max_delay now uses int_delays instead of the rounded delay_samples.
    # The result is the same concept — how many extra samples the output file
    # needs — but now correctly uses the floor rather than the rounded value.
    max_delay = int(np.max(int_delays))

    # [CHANGED] n_samples_total is slightly larger than the original to
    # accommodate the filter's own group delay (half_len samples of tail).
    # Original: n_samples_total = n_signal_samples + max_delay
    filter_group_delay  = frac_delay_half_len
    n_samples_total     = n_signal_samples + max_delay + filter_group_delay  # [CHANGED]

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Integer delay [samples]:", int_delays)                    # [CHANGED] was: delay_samples
    print("Fractional delay [samples]:", frac_delays.round(6))       # [CHANGED] new print
    print("Total output samples:", n_samples_total)

    # ── [CHANGED] pre-filter each satellite's baseband signal ────────────────
    #
    # Original code built the baseband chunk-by-chunk inside the loop below.
    # We now build the full baseband for each satellite once, apply the
    # fractional delay filter to it, and store the result in delayed_basebands.
    # The chunk loop below then slices into these pre-filtered arrays.
    #
    # Why pre-filter rather than filter inside the chunk loop?
    #   Simpler indexing.  The filter output has a well-defined offset
    #   (half_len samples) relative to the input, so it is easier to handle
    #   once up front than to manage overlap-save state across chunks.
    #   For very long chip streams (>~1 billion samples) you may prefer the
    #   overlap-save approach to save RAM — see the comment at the bottom of
    #   this function.
    #
    # Index relationship after filtering:
    #   filtered[k] corresponds to source sample (k - half_len).
    #   So to read source sample s, index filtered[s + half_len].
    # ─────────────────────────────────────────────────────────────────────────
    sample_idx_full = np.arange(n_signal_samples, dtype=np.int64)
    src_t_full      = sample_idx_full.astype(np.float64) / fs
    chip_phase_full = src_t_full * chip_rate
    chip_index_full = np.clip(
        np.floor(chip_phase_full).astype(np.int64), 0, n_chips - 1
    )

    delayed_basebands = []                                           # [CHANGED] new list
    for i, (chip_arr, h) in enumerate(zip(chips, frac_filters)):    # [CHANGED] new pre-filter loop
        baseband = chip_arr[chip_index_full]
        # fftconvolve with mode='full' gives length n_signal_samples + len(h) - 1.
        # Index [k] in the output corresponds to source sample (k - half_len).
        filtered = fftconvolve(baseband, h, mode='full').astype(np.float32)
        delayed_basebands.append(filtered)
    # ─────────────────────────────────────────────────────────────────────────

    with open(out_file, "wb") as f:
        for start in range(0, n_samples_total, chunk_size):
            stop = min(start + chunk_size, n_samples_total)

            sample_idx = np.arange(start, stop, dtype=np.int64)
            t = sample_idx.astype(np.float64) / fs

            carrier = np.cos(2 * np.pi * f_if * t).astype(np.float32)
            signal = np.zeros(stop - start, dtype=np.float32)

            for i in range(len(chips)):

                # ── [CHANGED] index into the pre-filtered baseband ────────────
                #
                # Original code:
                #     src_sample_idx = sample_idx - delay_samples[i]
                #     valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)
                #     ...
                #     chip_index = np.floor(chip_phase).astype(np.int64)
                #     baseband[valid] = chips[i][chip_index]
                #
                # Now we have a pre-filtered array where filtered[k] = signal
                # at source sample (k - half_len).  To get output sample `out`:
                #   source sample  = out - int_delays[i]
                #   filtered index = source_sample + half_len
                #                  = out - int_delays[i] + frac_delay_half_len
                #
                # The valid mask now guards against going outside the filtered
                # array rather than outside n_signal_samples directly.
                # ─────────────────────────────────────────────────────────────
                filt_idx = (                                         # [CHANGED] replaces src_sample_idx
                    sample_idx - int_delays[i] + frac_delay_half_len
                ).astype(np.int64)

                filtered  = delayed_basebands[i]                    # [CHANGED] replaces chip lookup
                valid     = (filt_idx >= 0) & (filt_idx < len(filtered))  # [CHANGED] guards filtered array

                if not np.any(valid):
                    continue

                # [CHANGED] read from pre-filtered array instead of chip array
                # Original built baseband from chips[i][chip_index] here.
                baseband_chunk = np.zeros(stop - start, dtype=np.float32)
                baseband_chunk[valid] = filtered[filt_idx[valid]]

                signal += amplitudes[i] * baseband_chunk * carrier  # [UNCHANGED]

            signal.tofile(f)  # [UNCHANGED]

    return n_samples_total  # [UNCHANGED]

    # ── RAM-efficient alternative (overlap-save) ──────────────────────────────
    # If holding all delayed_basebands in memory is too expensive, move the
    # convolution inside the chunk loop using overlap-save:
    #
    #   prev_tail = [np.zeros(len(h)-1) for h in frac_filters]
    #
    #   # inside per-satellite, per-chunk block:
    #   chunk_bb  = chip_arr[chip_index_chunk]
    #   padded    = np.concatenate([prev_tail[i], chunk_bb])
    #   filtered  = fftconvolve(padded, frac_filters[i], mode='full')
    #   out_chunk = filtered[half_len : half_len + (stop - start)]
    #   prev_tail[i] = chunk_bb[-(len(frac_filters[i])-1):]
    # ─────────────────────────────────────────────────────────────────────────
